# Diagnosing the flat-dataset / torus-latent collapse

Investigates a specific anomaly: training `TorusVAE` on `border_flat_torus`
(`make_plane_dataset`) reliably spreads the latent codes across the whole
torus, as expected -- but on `flat_torus` (`make_plane_dataset_old`,
"periodic flat plane"), the latent scatter can come out nearly empty (codes
collapsed into a small cluster instead of spread out), and whether that
happens seems to depend on hyperparameters that don't affect the border
case at all.

**The leading hypothesis, stated precisely so it can be tested rather than
just argued about:**

> `distributions.kl_wrapped_to_uniform` is mathematically INDEPENDENT of
> where `mu` is placed on the circle -- it only penalizes how confident
> (small-sigma) an individual encoding is, never whether different inputs
> get mapped to different locations. The ONLY term that can force different
> inputs onto different parts of the torus is reconstruction, and only if
> those inputs actually need visibly different decoder outputs.
>
> `border_flat_torus`'s images vary across the WHOLE 28x28 frame, so
> reconstruction alone forces the codes apart, regardless of hyperparameters.
> `flat_torus`'s images only vary within a 9x9-pixel window in the middle of
> the frame (see `datasets.py::make_plane_dataset_old`'s `lo=9, hi=18` at the
> default resolution) -- relabelled to span the full `[0, 2*pi)` "angle" for
> interface consistency, but barely distinguishable as IMAGES across that
> range. With reconstruction giving almost no pressure to differentiate
> codes, and KL giving none at all, collapsing onto a small cluster can be
> a genuinely low-loss outcome for this dataset specifically -- not obviously
> a bug, and sensitive to exactly the kind of hyperparameter that doesn't
> matter for the border case.

**Sections**
1. Setup
2. Config (sweep ranges -- edit epoch counts here for CPU vs GPU)
3. A concentration metric, to replace "eyeball the plot" with a number
4. **Direct proof of the KL-location-invariance claim** (no training needed,
   runs in under a second)
5. **Seed sweep**: is this a stochastic, borderline case?
6. **Anneal-length sweep**: does `sigma_anneal_epochs` actually matter here?
7. **Beta sweep**: does the KL weight change anything?
8. Side-by-side visual: the most collapsed vs. most spread run found
9. Summary


## 1. Setup

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())


In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Bachelorarbeit/projects_clean'  # <-- adjust if needed
else:
    PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "."

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({
    "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.fontsize": 11, "axes.titlesize": 14,
})

from vae_topology.datasets import make_plane_dataset, make_plane_dataset_old
from vae_topology.models.torus_vae import TorusVAE, train as train_torus
from vae_topology.distributions import wrap_angle, kl_wrapped_to_uniform
from vae_topology.diagnostics.latent_plots import diagnose

print("imports ok")


## 2. Config

`EPOCHS` below is intentionally lower than the 1000 you used when you first
saw this, so the whole sweep (roughly 20 training runs) finishes in a
reasonable time. This will likely still show the effect (or at least a
trend towards it) at a smaller scale; if you want to confirm it EXACTLY
matches your original 1000-epoch observation, raise `EPOCHS` (and expect the
whole notebook to take proportionally longer -- ~20 runs, not one).

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
EPOCHS = 300                    # your original report used 1000
ANNEAL_EPOCHS = 150             # baseline anneal window for the seed sweep
BETA = 1.0                      # baseline beta for the seed/anneal sweeps

SEEDS = [0, 1, 2, 3, 4]
ANNEAL_SWEEP = [50, 150, 300, EPOCHS]     # covers the "new default" (50) up to a full-run anneal
BETA_SWEEP = [0.25, 1.0, 2.0, 4.0]

print(f"device={DEVICE}  EPOCHS={EPOCHS}")


## 3. A concentration metric, instead of eyeballing the plot

The standard circular-statistics measure of "how spread out are these
angles": the mean resultant length

    R = | mean_i( exp(i * theta_i) ) |

`R = 0` means the angles are so spread out their vectors on the unit circle
cancel on average (healthy, uniform coverage). `R = 1` means every sample
landed at (nearly) the exact same angle (total collapse). This turns "the
plot looks nearly empty" into a single number you can compare across many
runs without staring at plots.

In [ ]:
def circular_concentration(mu):
    """mu: (N, 2) array of (theta1, theta2). Returns (R1, R2) in [0, 1]."""
    mu = np.asarray(mu)
    R = np.abs(np.exp(1j * mu).mean(axis=0))
    return float(R[0]), float(R[1])


def run_and_measure(data_fn, seed, epochs, sigma_anneal_epochs, beta, device=DEVICE):
    """Train TorusVAE once, return (metrics dict, mu, true_angles)."""
    model, data, angles, hist = train_torus(
        epochs=epochs, data_fn=data_fn, model_cls=TorusVAE, seed=seed,
        beta=beta, sigma_anneal_epochs=sigma_anneal_epochs, device=device,
        log_every=0,
    )
    model = model.double().eval()
    with torch.no_grad():
        mu = model.encode(data.double())[0].numpy()
    R1, R2 = circular_concentration(mu)
    last = hist[-1]
    metrics = dict(seed=seed, epochs=epochs, sigma_anneal_epochs=sigma_anneal_epochs,
                  beta=beta, R_theta1=R1, R_theta2=R2, R_mean=(R1 + R2) / 2,
                  loss=last["loss"], recon=last["recon"], kl=last["kl"],
                  sigma_x=last["sigma_x"])
    return metrics, mu, np.asarray(angles)


## 4. Direct proof: is the KL term really blind to WHERE `mu` is?

No training needed. Build two batches of reparametrized samples with the
SAME sigma: one where `mu` is spread uniformly across the whole circle, one
where every `mu` is jammed into the exact same point. If the hypothesis is
right, `kl_wrapped_to_uniform` should return (up to Monte Carlo noise from
the finite sample) the SAME value for both -- proving the KL term cannot be
what's spreading (or failing to spread) the codes in the real experiments
above; only reconstruction can be.

In [ ]:
torch.manual_seed(0)
N = 20000
sigma_val = 0.3
log_sigma = torch.full((N, 1), math.log(sigma_val), dtype=torch.float64)
eps = torch.randn(N, 1, dtype=torch.float64)

mu_spread = (torch.rand(N, 1, dtype=torch.float64) * 2 * math.pi - math.pi)
mu_collapsed = torch.zeros(N, 1, dtype=torch.float64)

theta_spread = wrap_angle(mu_spread + sigma_val * eps)
theta_collapsed = wrap_angle(mu_collapsed + sigma_val * eps)

kl_spread = kl_wrapped_to_uniform(theta_spread, mu_spread, log_sigma).item()
kl_collapsed = kl_wrapped_to_uniform(theta_collapsed, mu_collapsed, log_sigma).item()

print(f"KL, codes spread across the whole circle : {kl_spread:.6f}")
print(f"KL, codes ALL collapsed to one point     : {kl_collapsed:.6f}")
print(f"difference                                : {abs(kl_spread - kl_collapsed):.2e}"
     f"  (should be tiny -- Monte Carlo noise only, not a systematic gap)")


If the two numbers above came out close (they should, typically agreeing to
3+ significant figures at N=20000), that confirms the KL term genuinely
cannot tell these two situations apart -- so if `flat_torus` collapses and
`border_flat_torus` doesn't, the difference has to come from reconstruction,
which is exactly the "images barely differ across the labelled range"
argument above. The sweeps below test whether that argument actually
predicts the *sensitivity* you saw (hyperparameters flipping flat_torus
between collapsed and spread, while border_flat_torus stays put).

## 5. Seed sweep: is this a stochastic, borderline case?

Same everything except the seed, on both datasets. If `flat_torus`'s outcome
swings between collapsed (`R` near 1) and spread (`R` near 0) across seeds
while `border_flat_torus` stays consistently low-R regardless, that's strong
evidence this is a delicate/borderline case for `flat_torus`, not a
deterministic bug triggered by one specific change.

In [ ]:
rows = []
for seed in SEEDS:
    m, _, _ = run_and_measure(make_plane_dataset_old, seed, EPOCHS, ANNEAL_EPOCHS, BETA)
    m["dataset"] = "flat_torus"
    rows.append(m)
    print(f"  flat_torus    seed={seed}  R_mean={m['R_mean']:.3f}  kl={m['kl']:.3f}")

    m, _, _ = run_and_measure(make_plane_dataset, seed, EPOCHS, ANNEAL_EPOCHS, BETA)
    m["dataset"] = "border_flat_torus"
    rows.append(m)
    print(f"  border_flat_torus seed={seed}  R_mean={m['R_mean']:.3f}  kl={m['kl']:.3f}")

df_seed = pd.DataFrame(rows)
df_seed


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, g in df_seed.groupby("dataset"):
    ax.plot(g["seed"], g["R_mean"], "o-", label=name)
ax.axhline(0.0, color="0.7", lw=1)
ax.axhline(1.0, color="0.7", lw=1, ls="--")
ax.set_xlabel("seed"); ax.set_ylabel("mean resultant length R  (0=spread, 1=collapsed)")
ax.set_title("Seed sweep")
ax.legend()
fig.tight_layout()


## 6. Anneal-length sweep: does `sigma_anneal_epochs` actually matter here?

Fixed seed, varying only the anneal window -- from the new default (50)
up to annealing over the WHOLE run. Only run on `flat_torus`; section 5
already establishes `border_flat_torus`'s baseline behaviour, and it isn't
expected to move here (reconstruction pressure alone already keeps it
spread, independent of the noise schedule).

In [ ]:
rows = []
for ae in ANNEAL_SWEEP:
    m, _, _ = run_and_measure(make_plane_dataset_old, SEED, EPOCHS, ae, BETA)
    m["dataset"] = "flat_torus"
    rows.append(m)
    print(f"  sigma_anneal_epochs={ae:>4d}  R_mean={m['R_mean']:.3f}  kl={m['kl']:.3f}")

df_anneal = pd.DataFrame(rows)
df_anneal


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_anneal["sigma_anneal_epochs"], df_anneal["R_mean"], "o-", color="crimson")
ax.axhline(0.0, color="0.7", lw=1)
ax.axhline(1.0, color="0.7", lw=1, ls="--")
ax.set_xlabel("sigma_anneal_epochs"); ax.set_ylabel("mean resultant length R")
ax.set_title("Anneal-length sweep (flat_torus)")
fig.tight_layout()


## 7. Beta sweep: does the KL weight change anything?

Fixed seed and anneal window, varying `beta` (the KL weight in the ELBO).
Per the KL-location-invariance argument in section 4, `beta` scales a term
that doesn't care where codes are placed -- so if `flat_torus`'s collapse is
really about weak reconstruction pressure rather than KL pressure, `beta`
should have comparatively little effect on `R` here, unlike what you might
naively expect from "increase the KL weight to force more spread".

In [ ]:
rows = []
for b in BETA_SWEEP:
    m, _, _ = run_and_measure(make_plane_dataset_old, SEED, EPOCHS, ANNEAL_EPOCHS, b)
    m["dataset"] = "flat_torus"
    rows.append(m)
    print(f"  beta={b:>5.2f}  R_mean={m['R_mean']:.3f}  kl={m['kl']:.3f}  recon={m['recon']:.3f}")

df_beta = pd.DataFrame(rows)
df_beta


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_beta["beta"], df_beta["R_mean"], "o-", color="steelblue")
ax.axhline(0.0, color="0.7", lw=1)
ax.axhline(1.0, color="0.7", lw=1, ls="--")
ax.set_xlabel("beta"); ax.set_ylabel("mean resultant length R")
ax.set_title("Beta sweep (flat_torus)")
fig.tight_layout()


## 8. Side by side: the most collapsed vs. most spread run found

Pulls whichever two runs (across every sweep above) had the highest and
lowest `R_mean`, retrains them once more (cheap relative to the whole sweep),
and plots their actual latent scatter with `diagnose` -- so you can see with
your own eyes what a given `R` value actually looks like, calibrating the
number against the "nearly empty plot" you described.

In [ ]:
all_runs = pd.concat([df_seed[df_seed.dataset == "flat_torus"], df_anneal, df_beta],
                     ignore_index=True)
most_collapsed = all_runs.loc[all_runs["R_mean"].idxmax()]
most_spread = all_runs.loc[all_runs["R_mean"].idxmin()]

print("most collapsed run:\n", most_collapsed)
print("\nmost spread run:\n", most_spread)

for label, row in [("most_collapsed", most_collapsed), ("most_spread", most_spread)]:
    model, data, angles, hist = train_torus(
        epochs=int(row["epochs"]), data_fn=make_plane_dataset_old,
        model_cls=TorusVAE, seed=int(row["seed"]), beta=float(row["beta"]),
        sigma_anneal_epochs=int(row["sigma_anneal_epochs"]), device=DEVICE, log_every=0,
    )
    diagnose(model, data, angles, save=f"{label}.pdf")
    print(f"{label}: R_mean={row['R_mean']:.3f} -> see {label}_torus.pdf / {label}_domain.pdf")


## 9. Summary

Re-run this cell after the sweeps above to get a quick printed verdict.

In [ ]:
print("=== seed sweep ===")
print(df_seed.groupby("dataset")["R_mean"].agg(["mean", "std", "min", "max"]))

print("\n=== anneal-length sweep (flat_torus) ===")
print(df_anneal[["sigma_anneal_epochs", "R_mean", "kl"]])

print("\n=== beta sweep (flat_torus) ===")
print(df_beta[["beta", "R_mean", "kl"]])

print("""
How to read this:
  - If border_flat_torus's R stays low and tight across all seeds (section 5)
    while flat_torus's R varies a lot -- that confirms flat_torus is the
    fragile one, consistent with the reconstruction-pressure argument.
  - If flat_torus's R changes a lot across ANNEAL_SWEEP (section 6) -- the
    noise schedule genuinely matters for this dataset, and picking a
    specific sigma_anneal_epochs is a real, usable lever.
  - If flat_torus's R barely changes across BETA_SWEEP (section 7) -- that's
    the KL-invariance argument (section 4) playing out in the full model,
    not just the isolated KL formula.
""")
